# KL Grading: YOLO ROI CNN Comparison

This is a controlled, validation-only **CNN-only** model comparison for Kellgren-Lawrence grades 0-4. It holds the production YOLOv8 ROI contract constant.

Candidates: **EfficientNetV2-S** (CNN baseline), **HRNet-W32** (high-resolution CNN), and **ConvNeXt-Tiny** (modern convolutional model). Each run writes the YOLO checkpoint SHA-256 and ROI manifest metadata into its classifier checkpoint.

Do not use a repeatedly inspected test split for selection. The notebook trains on `train`, chooses by `val`, and leaves `test` locked. It never center-crops an ROI: marginal osteophytes must remain visible.

## Required ROI Contract

Build the ROI dataset once with the same detector that will be used at inference:

```bash
python3 scripts/build_production_yolo_roi_dataset.py \
  --full-images /path/to/full_radiographs \
  --labels-root /path/to/labeled_knee_crops \
  --yolo-checkpoint checkpoints/yolov8/2026-07-26_20-49-25_joint_detection/best.pt \
  --output data/processed/yolo_roi_v1 \
  --confidence 0.45
```

That builder preserves exact integer `xyxy` detector crops and records the source, ROI, and detector hashes in `manifest.csv` and `metadata.json`. Do not replace the detector, threshold, coordinate rounding, laterality convention, or ROI crop policy between candidate models.

In [ ]:
import csv
import hashlib
import json
import random
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from PIL import Image, ImageOps
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'main.py').exists():
    REPO_ROOT = REPO_ROOT.parent

YOLO_CHECKPOINT = REPO_ROOT / 'checkpoints/yolov8/2026-07-26_20-49-25_joint_detection/best.pt'
ROI_DATASET_ROOT = REPO_ROOT / 'data/processed/yolo_roi_v1'
ROI_MANIFEST = ROI_DATASET_ROOT / 'manifest.csv'
ROI_METADATA = ROI_DATASET_ROOT / 'metadata.json'
RUN_ROOT = REPO_ROOT / 'checkpoints/architecture_benchmark'

IMAGE_SIZE = 384
BATCH_SIZE = 12
NUM_WORKERS = 4
EPOCHS = 30
PATIENCE = 7
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
LOSS_MODE = 'ce'  # 'ce', 'ordinal', or 'mixed'
ORDINAL_WEIGHT = 0.25
YOLO_CONFIDENCE = 0.45
YOLO_IOU = 0.70
PRETRAINED = True

EXPERIMENTS = [
    {'name': 'efficientnetv2_s', 'timm_name': 'tf_efficientnetv2_s', 'family': 'cnn', 'enabled': True},
    {'name': 'hrnet_w32', 'timm_name': 'hrnet_w32', 'family': 'high_resolution_cnn', 'enabled': True},
    {'name': 'convnext_tiny', 'timm_name': 'convnext_tiny', 'family': 'modern_convolutional', 'enabled': True},
]

assert LOSS_MODE in {'ce', 'ordinal', 'mixed'}
print(f'Repository: {REPO_ROOT}')
print(f'YOLO checkpoint: {YOLO_CHECKPOINT}')
print(f'ROI manifest: {ROI_MANIFEST}')

## Validate the YOLOv8 Checkpoint

A usable checkpoint is a local Ultralytics **detection** `.pt` model trained for a knee-joint ROI class. A single ROI class is preferred; multiple classes are rejected here because silent class selection creates an unstable crop contract. The important downstream requirement is high recall and anatomically complete crops, not detector mAP alone.

The repository's current detector record is YOLOv8n, 640-pixel input, 100 epochs, batch size 16, validation precision 0.99952, recall 0.99145, mAP50 0.99500, and mAP50-95 0.90477. Revalidate it on the intended image domain before a clinical claim.

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def validate_yolo_contract(checkpoint: Path, metadata_path: Path) -> dict:
    if not checkpoint.is_file() or checkpoint.suffix.lower() != '.pt':
        raise FileNotFoundError(f'Expected a YOLO detection .pt checkpoint: {checkpoint}')
    try:
        from ultralytics import YOLO
    except ImportError as error:
        raise RuntimeError('Install the project requirements, including ultralytics, before running this cell.') from error

    detector = YOLO(str(checkpoint))
    names = dict(detector.names)
    if detector.task != 'detect':
        raise ValueError(f'Expected YOLO task=detect, found {detector.task!r}.')
    if len(names) != 1:
        raise ValueError(f'Expected one knee-joint class, found {names}. Pin an explicit class policy before use.')

    checkpoint_hash = sha256_file(checkpoint)
    metadata = json.loads(metadata_path.read_text()) if metadata_path.is_file() else {}
    if metadata:
        if metadata.get('yolo_checkpoint_sha256') != checkpoint_hash:
            raise ValueError('ROI metadata was made with a different YOLO checkpoint. Rebuild the ROI dataset.')
        if float(metadata.get('yolo_confidence_threshold', -1)) != YOLO_CONFIDENCE:
            raise ValueError('ROI metadata uses a different confidence threshold. Rebuild the ROI dataset.')
        expected_contract = 'exact integer xyxy YOLO crop; no CLAHE, padding, resize, or mirroring baked in'
        if metadata.get('crop_contract') != expected_contract:
            raise ValueError('ROI metadata has an incompatible crop contract.')

    result = {
        'checkpoint': str(checkpoint),
        'sha256': checkpoint_hash,
        'task': detector.task,
        'class_names': names,
        'imgsz': 640,
        'confidence': YOLO_CONFIDENCE,
        'iou': YOLO_IOU,
        'roi_metadata_verified': bool(metadata),
    }
    print(json.dumps(result, indent=2))
    return result


YOLO_CONTRACT = validate_yolo_contract(YOLO_CHECKPOINT, ROI_METADATA)

## Load a Patient-Safe Manifest

`manifest.csv` must contain `patient_id`, `knee_side`, `split`, `grade`, and `output_roi`. Right ROIs are mirrored only at classifier load time to create a canonical orientation. The detector crop itself is never modified. The fixed preprocessing is: raw detector crop -> optional right-knee canonicalization -> mild CLAHE -> square padding -> resize -> ImageNet normalization. There is no center crop.

In [ ]:
REQUIRED_COLUMNS = {'patient_id', 'knee_side', 'split', 'grade', 'output_roi'}
manifest = pd.read_csv(ROI_MANIFEST, dtype={'patient_id': str})
missing = REQUIRED_COLUMNS - set(manifest.columns)
if missing:
    raise ValueError(f'Manifest is missing columns: {sorted(missing)}')
manifest['grade'] = manifest['grade'].astype(int)
if not set(manifest['grade']).issubset(set(range(5))):
    raise ValueError('KL grades must be integers 0 through 4.')
if not {'train', 'val'}.issubset(set(manifest['split'])):
    raise ValueError('The architecture comparison requires train and val rows.')

patient_split_counts = manifest.groupby('patient_id')['split'].nunique()
leaked_patients = patient_split_counts[patient_split_counts > 1]
if not leaked_patients.empty:
    raise ValueError(f'Patient leakage across splits: {leaked_patients.index[:10].tolist()}')

for _, row in manifest.iterrows():
    if not (ROI_DATASET_ROOT / row.output_roi).is_file():
        raise FileNotFoundError(f'Missing ROI image: {ROI_DATASET_ROOT / row.output_roi}')

print(manifest.groupby(['split', 'grade']).size().unstack(fill_value=0))
print(f'Unique patients: {manifest.patient_id.nunique()}')


def clahe_rgb(image: Image.Image) -> Image.Image:
    array = np.asarray(image.convert('RGB'))
    lab = cv2.cvtColor(array, cv2.COLOR_RGB2LAB)
    lab[:, :, 0] = cv2.createCLAHE(clipLimit=1.25, tileGridSize=(8, 8)).apply(lab[:, :, 0])
    return Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB))


def square_pad(image: Image.Image) -> Image.Image:
    width, height = image.size
    side = max(width, height)
    left = (side - width) // 2
    top = (side - height) // 2
    return ImageOps.expand(image, border=(left, top, side - width - left, side - height - top), fill=0)


class KLROIDataset(Dataset):
    def __init__(self, rows: pd.DataFrame, train: bool):
        self.rows = rows.reset_index(drop=True)
        self.train = train
        augmentation = [
            transforms.RandomRotation(7, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.RandomAffine(degrees=0, translate=(0.03, 0.03), scale=(0.95, 1.05)),
        ] if train else []
        self.tensor_transform = transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BILINEAR),
            *augmentation,
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ])

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]
        image = Image.open(ROI_DATASET_ROOT / row.output_roi).convert('RGB')
        if row.knee_side == 'right':
            image = ImageOps.mirror(image)
        image = square_pad(clahe_rgb(image))
        return self.tensor_transform(image), int(row.grade), str(row.patient_id)


train_rows = manifest[manifest.split == 'train'].copy()
val_rows = manifest[manifest.split == 'val'].copy()
train_loader = DataLoader(KLROIDataset(train_rows, train=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(KLROIDataset(val_rows, train=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

## Shared Training Protocol

The comparison uses ImageNet initialization, AdamW, cosine learning-rate decay, mixed precision, early stopping on validation quadratic weighted kappa, and the same ROI transforms for every architecture. Start with `LOSS_MODE = 'ce'` to isolate architecture. Repeat the finalists with the ordinal or mixed loss as a separate ablation; do not change architecture and loss in the same experiment.

In [ ]:
ORDINAL_COST = torch.tensor([
    [0.0, 1.0, 4.0, 9.0, 16.0],
    [1.0, 0.0, 1.0, 4.0, 9.0],
    [4.0, 1.0, 0.0, 1.0, 4.0],
    [9.0, 4.0, 1.0, 0.0, 1.0],
    [16.0, 9.0, 4.0, 1.0, 0.0],
], dtype=torch.float32, device=DEVICE)


class KLLoss(nn.Module):
    def __init__(self, mode: str, ordinal_weight: float):
        super().__init__()
        self.mode = mode
        self.ordinal_weight = ordinal_weight
        self.cross_entropy = nn.CrossEntropyLoss(label_smoothing=0.05)

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        ce = self.cross_entropy(logits, target)
        expected_distance = (torch.softmax(logits, dim=1) * ORDINAL_COST[target]).sum(dim=1).mean()
        if self.mode == 'ce':
            return ce
        if self.mode == 'ordinal':
            return expected_distance
        return ce + self.ordinal_weight * expected_distance


def build_model(spec: dict) -> nn.Module:
    kwargs = {'pretrained': PRETRAINED, 'num_classes': 5}
    try:
        return timm.create_model(spec['timm_name'], img_size=IMAGE_SIZE, **kwargs)
    except TypeError:
        return timm.create_model(spec['timm_name'], **kwargs)


def evaluate(model: nn.Module, loader: DataLoader, criterion: nn.Module) -> dict:
    model.eval()
    losses, labels, predictions = [], [], []
    with torch.no_grad():
        for images, targets, _ in loader:
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            with autocast(enabled=torch.cuda.is_available()):
                logits = model(images)
                loss = criterion(logits, targets)
            losses.append(float(loss.item()) * len(targets))
            labels.extend(targets.cpu().tolist())
            predictions.extend(logits.argmax(dim=1).cpu().tolist())
    return {
        'loss': sum(losses) / len(labels),
        'accuracy': accuracy_score(labels, predictions),
        'macro_f1': f1_score(labels, predictions, average='macro', zero_division=0),
        'qwk': cohen_kappa_score(labels, predictions, weights='quadratic'),
        'n': len(labels),
    }

In [ ]:
@dataclass
class RunMetadata:
    architecture: str
    timm_name: str
    family: str
    started_at_utc: str
    yolo_contract: dict
    roi_manifest: str
    roi_manifest_sha256: str
    image_size: int
    loss_mode: str
    optimizer: str
    learning_rate: float
    weight_decay: float
    epochs_requested: int


def train_candidate(spec: dict) -> dict:
    run_id = datetime.now(timezone.utc).strftime('%Y-%m-%d_%H-%M-%S_%f_UTC')
    run_dir = RUN_ROOT / f'{run_id}_{spec["name"]}_{LOSS_MODE}_comparison'
    run_dir.mkdir(parents=True, exist_ok=False)
    model = build_model(spec).to(DEVICE)
    criterion = KLLoss(LOSS_MODE, ORDINAL_WEIGHT)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = GradScaler(enabled=torch.cuda.is_available())
    metadata = RunMetadata(
        architecture=spec['name'], timm_name=spec['timm_name'], family=spec['family'],
        started_at_utc=datetime.now(timezone.utc).isoformat(), yolo_contract=YOLO_CONTRACT,
        roi_manifest=str(ROI_MANIFEST), roi_manifest_sha256=sha256_file(ROI_MANIFEST),
        image_size=IMAGE_SIZE, loss_mode=LOSS_MODE, optimizer='AdamW',
        learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY, epochs_requested=EPOCHS,
    )
    (run_dir / 'metadata.json').write_text(json.dumps(asdict(metadata), indent=2))

    best_qwk, stale_epochs, history = -float('inf'), 0, []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss, seen = 0.0, 0
        for images, targets, _ in tqdm(train_loader, desc=f'{spec["name"]} epoch {epoch}/{EPOCHS}', leave=False):
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with autocast(enabled=torch.cuda.is_available()):
                logits = model(images)
                loss = criterion(logits, targets)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            running_loss += float(loss.item()) * len(targets)
            seen += len(targets)
        scheduler.step()
        metrics = evaluate(model, val_loader, criterion)
        metrics.update({'epoch': epoch, 'train_loss': running_loss / seen, 'lr': optimizer.param_groups[0]['lr']})
        history.append(metrics)
        print(json.dumps(metrics, sort_keys=True))
        if metrics['qwk'] > best_qwk:
            best_qwk, stale_epochs = metrics['qwk'], 0
            torch.save({'model_state_dict': model.state_dict(), 'metadata': asdict(metadata), 'best_val': metrics}, run_dir / 'best_model.pth')
        else:
            stale_epochs += 1
            if stale_epochs >= PATIENCE:
                print(f'Early stopping after {epoch} epochs.')
                break

    pd.DataFrame(history).to_csv(run_dir / 'history.csv', index=False)
    best = max(history, key=lambda row: row['qwk'])
    best.update({'architecture': spec['name'], 'family': spec['family'], 'run_dir': str(run_dir)})
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return best


results = []
for spec in EXPERIMENTS:
    if spec['enabled']:
        results.append(train_candidate(spec))
results_frame = pd.DataFrame(results).sort_values(['qwk', 'macro_f1'], ascending=False)
results_frame.to_csv(RUN_ROOT / 'architecture_comparison_summary.csv', index=False)
results_frame

## CAM Audit for the Validation Winner

For every candidate in this CNN-only notebook, use the final convolutional layer for Grad-CAM. Review at least 20 validation cases across grades, including osteophyte margins, joint-space narrowing, sclerosis, and failure cases. Do not compare CAMs from different ROI crops.

In [ ]:
# Optional dependency for CAM auditing: python3 -m pip install grad-cam
# This cell does not block training when the optional package is absent.
try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
except ImportError:
    GradCAM = None
    show_cam_on_image = None
    print('Install grad-cam after selecting a winner to run the CAM renderer.')


def candidate_cam_layers(model: nn.Module):
    return [name for name, module in model.named_modules() if isinstance(module, nn.Conv2d)][-12:]


def render_gradcam(model, target_layer, image_tensor):
    if GradCAM is None:
        raise RuntimeError('Install grad-cam before rendering a CAM.')
    cam = GradCAM(model=model, target_layers=[target_layer])
    grayscale_cam = cam(input_tensor=image_tensor.unsqueeze(0).to(DEVICE))[0]
    image = image_tensor.permute(1, 2, 0).cpu().numpy()
    image = image * np.array((0.229, 0.224, 0.225)) + np.array((0.485, 0.456, 0.406))
    overlay = show_cam_on_image(np.clip(image, 0, 1).astype(np.float32), grayscale_cam, use_rgb=True)
    plt.figure(figsize=(6, 6))
    plt.imshow(overlay)
    plt.axis('off')
    return grayscale_cam

# Example after loading the winner checkpoint:
# print(candidate_cam_layers(winner_model))
# Select the last spatial convolutional layer after inspecting this list.
# Never use this automatic list as the final target-layer decision.

## Decision Rule

Promote at most two candidates for a clean, locked evaluation. Primary criterion: validation QWK. Secondary criteria: macro-F1, grade-1 recall, calibration, and CAM anatomy audit. A model with approximately 70% accuracy can be acceptable only when its held-out QWK, per-grade errors, detector-crop robustness, and localization are defensible. Record the exact executed notebook, timestamped run directory, all metrics, and CAM figures under `docs/report/<model>/`.